## Script Used to Filter Out Invalid IaC Templates / Validate IaC Templates

This script is used to filter out invalid IaC templates / validate IaC templates for the new dataset collection pipeline

CloudFormation Validation
- Use yaml / json loader to load the IaC template then check if there is any resource exists with Type value in "xxx::xxx::xxx" format (i.e., with .get("Resources", "")). If there is not resource, label the template as invalid.

ARM Validation
- Uses the schema as the keyword check if its value ends with either of [deploymentTemplate.json#, subscriptionDeploymentTemplate.json#, managementGroupDeploymentTemplate.json#, tenantDeploymentTemplate.json#]. If not label the template as invalid.
- Use the json loader and check if there is any resource exists. If not, label the template as invalid.

Terraform Validation 
- Use HCL2 loader to load the IaC template and then check if there is any resource exists
- Load the template to check the provider. If the provider is not in [aws, azurerm, gcp], label the template in the dataset and is invalid. Also save the provider/s in the dataset.

Bicep Validation
- Use `pycep` to parse and check if there is any resource exists. If not, label the template as invalid.

Bicep Collection & Filtering
- Collect all .bicep file from repo.
- Scan each file for module declarations and record the relative path in csv file
	- Classify as local (./…), registry (br:…), and template spec (ts:…) and store in three separate columns. Keep empty (NaN) if there is not such value for a column. 
- Scan each file for (loadFileAsBase64, loadTextContent, loadJsonContent, loadYamlContent), extract and store the file path in the csv file's column. 
- Group the parent & all referenced module files with local modules and the html_url to check for the same repo. Store the reference module file in the correct file path (create new directory if needed). Filter out Bicep Group that contain template spec (ts:…).
- Create stubs for external files.
- During parsing: Recreate the directory structure in a temporary folder for the modules then run bicep build on the parent file.
	- Note to get the nested module chains during parsing (i.e., a module referencing another module and the module referencing other modules)

In [1]:
import os
import re
import json
import pycep
import pandas as pd
from pathlib import Path


# Set working directory to project root (parent of data/)
PROJECT_ROOT = Path(os.path.abspath('')).parent if Path(os.path.abspath('')).name == 'data' else Path(os.path.abspath(''))
os.chdir(PROJECT_ROOT)
print(f"Working directory: {os.getcwd()}")

Working directory: d:\Personal Storage\1 - PhD\0_Projects\1_IR\code


### CloudFormation Validation

In [36]:
CFN_TAGS = [
    '!Ref', '!Sub', '!GetAtt', '!Join', '!Select', '!Split', '!Equals', '!If',
    '!FindInMap', '!GetAZs', '!Base64', '!Cidr', '!Transform', '!ImportValue',
    '!Not', '!And', '!Or', '!Condition', '!ForEach', '!ValueOf',
    '!Rain::Embed', '!Rain::Module',
]


def _load_cfn_template(content: str) -> dict:
    """
    Load a CloudFormation template from string content.
    Uses the same strategy as the CloudFormation parser:
      - JSON loader if the content starts with '{'.
      - Custom YAML loader with CFN intrinsic-function tag constructors otherwise.
    """
    import yaml

    if content.lstrip().startswith('{'):
        return json.loads(content)

    class _CfnLoader(yaml.SafeLoader):
        pass

    def _construct_cfn_tag(loader, node):
        tag_name = node.tag[1:]
        if isinstance(node, yaml.ScalarNode):
            return {tag_name: node.value}
        elif isinstance(node, yaml.SequenceNode):
            return {tag_name: loader.construct_sequence(node)}
        elif isinstance(node, yaml.MappingNode):
            return {tag_name: loader.construct_mapping(node)}

    for tag in CFN_TAGS:
        _CfnLoader.add_constructor(tag, _construct_cfn_tag)

    return yaml.load(content, Loader=_CfnLoader)


def validate_cloudformation(input_csv_path: str, output_csv_path: str, encoding: str = 'utf-8') -> pd.DataFrame:
    """
    Validate CloudFormation templates from a CSV file.

    Checks:
      1. Template can be loaded as YAML (with CFN intrinsic function tags) or JSON.
      2. 'Resources' key exists and is a non-empty dict.
      3. At least one resource has a Type matching the 'xxx::xxx::xxx' pattern.

    Adds columns: is_valid (bool), invalid_reason (str).
    """
    df = pd.read_csv(input_csv_path, encoding=encoding)
    cfn_mask = df['template_language'].str.lower().isin(['cloudformation', 'cfn'])
    cfn_df = df[cfn_mask].copy()

    # type_pattern = re.compile(r'^\w+::\w+::\w+')
    type_pattern = re.compile(r'^\w+::\w+')

    is_valid_list = []
    invalid_reason_list = []

    for _, row in cfn_df.iterrows():
        template_path = row['template_path']

        if not os.path.exists(template_path):
            is_valid_list.append(False)
            invalid_reason_list.append(f"File not found: {template_path}")
            continue

        try:
            with open(template_path, 'r', encoding='utf-8-sig') as f:
                content = f.read()

            data = _load_cfn_template(content)

            if not isinstance(data, dict):
                is_valid_list.append(False)
                invalid_reason_list.append("Template root is not a dictionary")
                continue

            resources = data.get('Resources', {})
            if not resources or not isinstance(resources, dict):
                is_valid_list.append(False)
                invalid_reason_list.append("No 'Resources' section found or it is empty")
                continue

            has_valid_type = any(
                isinstance(res, dict) and type_pattern.match(str(res.get('Type', '')))
                for res in resources.values()
            )
            if not has_valid_type:
                is_valid_list.append(False)
                invalid_reason_list.append("No resource with valid Type (xxx::xxx::xxx) found")
                continue

            is_valid_list.append(True)
            invalid_reason_list.append("")

        except Exception as e:
            is_valid_list.append(False)
            invalid_reason_list.append(f"Load error: {str(e)}")

    cfn_df['is_valid'] = is_valid_list
    cfn_df['invalid_reason'] = invalid_reason_list

    cfn_df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')

    total = len(cfn_df)
    valid = sum(is_valid_list)
    invalid = total - valid
    print(f"[CloudFormation] {valid}/{total} valid ({valid/total*100:.1f}%), {invalid} invalid")
    if invalid > 0:
        reasons = cfn_df[cfn_df['is_valid'] == False]['invalid_reason'].value_counts()
        for reason, count in reasons.items():
            print(f"  - {reason}: {count}")

    return cfn_df

In [37]:
with open("data/cloudformation_collected_templates_new/template_00580_buckets.yml", 'r', encoding='utf-8-sig') as f:
    content = f.read()

data = _load_cfn_template(content)
data

{'AWSTemplateFormatVersion': '2010-09-09-OC',
 'Definitions': [{'Type': 'AWS::S3::Bucket'}],
 'Resources': {'Bucket': {'Type': 'AWS::S3::Bucket'}},
 'Outputs': {'BucketName': {'Value': {'GetAtt': 'Bucket.Arn'},
   'Export': {'Name': 'BucketArn'}}}}

### ARM Validation

In [38]:
VALID_ARM_SCHEMAS = (
    'deploymentTemplate.json#',
    'subscriptionDeploymentTemplate.json#',
    'managementGroupDeploymentTemplate.json#',
    'tenantDeploymentTemplate.json#',
)


def validate_arm(input_csv_path: str, output_csv_path: str, encoding: str = 'utf-8') -> pd.DataFrame:
    """
    Validate ARM templates from a CSV file.

    Checks:
      1. Template can be loaded as JSON.
      2. '$schema' ends with a valid deployment template schema.
      3. 'resources' key exists and is a non-empty list.

    Adds columns: is_valid (bool), invalid_reason (str), schema_type (str).
    """
    df = pd.read_csv(input_csv_path, encoding=encoding)
    arm_mask = df['template_language'].str.lower().isin(['arm', 'azure'])
    arm_df = df[arm_mask].copy()

    is_valid_list = []
    invalid_reason_list = []
    schema_type_list = []

    for _, row in arm_df.iterrows():
        template_path = row['template_path']

        if not os.path.exists(template_path):
            is_valid_list.append(False)
            invalid_reason_list.append(f"File not found: {template_path}")
            schema_type_list.append("")
            continue

        try:
            with open(template_path, 'r', encoding='utf-8-sig') as f:
                data = json.load(f)

            if not isinstance(data, dict):
                is_valid_list.append(False)
                invalid_reason_list.append("Template root is not a dictionary")
                schema_type_list.append("")
                continue

            schema = data.get('$schema', '')
            if not any(schema.endswith(s) for s in VALID_ARM_SCHEMAS):
                is_valid_list.append(False)
                invalid_reason_list.append(f"Invalid or missing $schema: {schema}")
                schema_type_list.append(schema)
                continue

            resources = data.get('resources', [])
            if not resources or not isinstance(resources, list):
                is_valid_list.append(False)
                invalid_reason_list.append("No 'resources' section found or it is empty")
                schema_type_list.append(schema)
                continue

            is_valid_list.append(True)
            invalid_reason_list.append("")
            schema_type_list.append(schema)

        except json.JSONDecodeError as e:
            is_valid_list.append(False)
            invalid_reason_list.append(f"JSON parse error: {str(e)}")
            schema_type_list.append("")
        except Exception as e:
            is_valid_list.append(False)
            invalid_reason_list.append(f"Load error: {str(e)}")
            schema_type_list.append("")

    arm_df['is_valid'] = is_valid_list
    arm_df['invalid_reason'] = invalid_reason_list
    arm_df['schema_type'] = schema_type_list

    arm_df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')

    total = len(arm_df)
    valid = sum(is_valid_list)
    invalid = total - valid
    print(f"[ARM] {valid}/{total} valid ({valid/total*100:.1f}%), {invalid} invalid")
    if invalid > 0:
        reasons = arm_df[arm_df['is_valid'] == False]['invalid_reason'].value_counts()
        for reason, count in reasons.items():
            print(f"  - {reason}: {count}")

    return arm_df

### Terraform Validation

In [39]:
def process_tf_csv(csv_file: str) -> None:
    df = pd.read_csv(csv_file)
    
    def build_file_paths(row):
        filenames = [f.strip() for f in str(row['file_names']).split(';')]
        paths = [f"{row['template_path']}/{filename}" for filename in filenames if filename]
        return '|'.join(paths)
    
    df['file_paths'] = df.apply(build_file_paths, axis=1)
    df.to_csv(csv_file, index=False)

In [40]:
KNOWN_PROVIDERS = {'aws', 'azurerm', 'google'}


def validate_terraform(input_csv_path: str, output_csv_path: str, encoding: str = 'utf-8') -> pd.DataFrame:
    """
    Validate Terraform templates (directory-level modules) from a CSV file.

    Each row represents a module group where:
      - template_path  : directory containing the .tf files
      - file_names     : semicolon-separated list of .tf filenames inside that directory

    Checks:
      1. All listed files exist on disk.
      2. The merged HCL content can be loaded with the hcl2 parser.
      3. At least one 'resource' block exists across all files in the module.
      4. Providers are extracted from resource type prefixes (e.g., aws_instance -> aws).
         If none of the detected providers are in KNOWN_PROVIDERS, the template is labelled
         but NOT marked invalid (it may still be a valid template for another provider).

    Adds columns: is_valid (bool), invalid_reason (str), providers (str, comma-separated).
    """
    import hcl2

    df = pd.read_csv(input_csv_path, encoding=encoding)
    tf_mask = df['template_language'].str.lower().isin(['terraform', 'tf', 'hcl'])
    tf_df = df[tf_mask].copy()

    is_valid_list = []
    invalid_reason_list = []
    providers_list = []

    for _, row in tf_df.iterrows():
        group_dir = str(row.get('template_path', ''))
        file_names_raw = str(row.get('file_names', ''))
        file_paths = [
            os.path.join(group_dir, fn.strip())
            for fn in file_names_raw.split(';') if fn.strip()
        ]

        if not file_paths:
            is_valid_list.append(False)
            invalid_reason_list.append("No files listed in file_names")
            providers_list.append("")
            continue

        missing = [p for p in file_paths if not os.path.exists(p)]
        if missing:
            is_valid_list.append(False)
            invalid_reason_list.append(f"File(s) not found: {', '.join(missing)}")
            providers_list.append("")
            continue

        try:
            merged_data = {}
            for fp in file_paths:
                with open(fp, 'r', encoding='utf-8-sig') as f:
                    data = hcl2.load(f)
                for key, value in data.items():
                    merged_data.setdefault(key, []).extend(value if isinstance(value, list) else [value])

            resources = merged_data.get('resource', [])
            modules = merged_data.get('module', [])
            if not resources and not modules:
                is_valid_list.append(False)
                invalid_reason_list.append("No 'resource' block found")
                providers_list.append("")
                continue

            # Extract providers from resource type prefixes (e.g. "aws_instance" -> "aws")
            detected_providers = set()
            for res_block in resources:
                if isinstance(res_block, dict):
                    for resource_type in res_block.keys():
                        prefix = resource_type.split('_')[0] if '_' in resource_type else resource_type
                        detected_providers.add(prefix)

            providers_str = ','.join(sorted(detected_providers))
            providers_list.append(providers_str)

            has_known = detected_providers & KNOWN_PROVIDERS
            if not has_known:
                is_valid_list.append(True)
                invalid_reason_list.append(f"No known cloud provider found (detected: {providers_str})")
                continue

            is_valid_list.append(True)
            invalid_reason_list.append("")

        except Exception as e:
            is_valid_list.append(False)
            invalid_reason_list.append(f"HCL2 parse error: {str(e)}")
            providers_list.append("")

    tf_df['is_valid'] = is_valid_list
    tf_df['invalid_reason'] = invalid_reason_list
    tf_df['providers'] = providers_list

    tf_df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')

    total = len(tf_df)
    valid = sum(is_valid_list)
    invalid = total - valid
    print(f"[Terraform] {valid}/{total} valid ({valid/total*100:.1f}%), {invalid} invalid")
    if invalid > 0:
        reasons = tf_df[tf_df['is_valid'] == False]['invalid_reason'].value_counts()
        for reason, count in reasons.items():
            print(f"  - {reason}: {count}")

    provider_counts = tf_df[tf_df['providers'] != '']['providers'].value_counts()
    print(f"  Provider distribution (top 10):")
    for provider, count in provider_counts.head(10).items():
        print(f"    {provider}: {count}")

    return tf_df

### Bicep Validation

In [2]:
MODULE_LOCAL_RE = re.compile(r"module\s+\w+\s+'(\./[^']+\.bicep)'\s*=")
MODULE_REGISTRY_RE = re.compile(r"module\s+\w+\s+'(br:[^']+)'\s*=")
MODULE_TEMPLATE_SPEC_RE = re.compile(r"module\s+\w+\s+'(ts:[^']+)'\s*=")
LOAD_FILE_RE = re.compile(
    r"(?:loadFileAsBase64|loadTextContent|loadJsonContent|loadYamlContent)\(\s*'([^']+)'\s*\)"
)


def _scan_bicep_file(file_path: str) -> dict:
    """
    Scan a single Bicep file for module declarations and external file references.
    Returns a dict with lists of local modules, registry modules, template spec modules,
    and external file paths.
    """
    try:
        with open(file_path, 'r', encoding='utf-8-sig') as f:
            content = f.read()
    except Exception:
        return {'local_modules': [], 'registry_modules': [], 'template_spec_modules': [], 'external_files': []}

    return {
        'local_modules': MODULE_LOCAL_RE.findall(content),
        'registry_modules': MODULE_REGISTRY_RE.findall(content),
        'template_spec_modules': MODULE_TEMPLATE_SPEC_RE.findall(content),
        'external_files': LOAD_FILE_RE.findall(content),
    }


def validate_bicep(input_csv_path: str, output_csv_path: str, encoding: str = 'utf-8') -> pd.DataFrame:
    """
    Validate Bicep templates from a CSV file.

    Checks:
      1. Template can be parsed by pycep.
      2. At least one 'resource' exists in the parsed output.

    Also scans each file for:
      - Local module references (./...)
      - Registry module references (br:...)
      - Template spec module references (ts:...)
      - External file references (loadFileAsBase64, loadTextContent, etc.)

    Adds columns:
      is_valid (bool), invalid_reason (str),
      local_modules (pipe-separated), registry_modules (pipe-separated),
      template_spec_modules (pipe-separated), external_files (pipe-separated).
    """
    from pycep import BicepParser

    df = pd.read_csv(input_csv_path, encoding=encoding)
    bicep_mask = df['template_language'].str.lower().isin(['bicep', 'bicep_template'])
    bicep_df = df[bicep_mask].copy()

    pycep_parser = BicepParser()

    is_valid_list = []
    invalid_reason_list = []
    local_modules_list = []
    registry_modules_list = []
    ts_modules_list = []
    external_files_list = []

    for _, row in bicep_df.iterrows():
        template_path = row['template_path']

        if not os.path.exists(template_path):
            is_valid_list.append(False)
            invalid_reason_list.append(f"File not found: {template_path}")
            local_modules_list.append("")
            registry_modules_list.append("")
            ts_modules_list.append("")
            external_files_list.append("")
            continue

        # Scan for modules and external files (regex-based, works even if pycep fails)
        scan = _scan_bicep_file(template_path)
        local_modules_list.append('|'.join(scan['local_modules']) if scan['local_modules'] else "")
        registry_modules_list.append('|'.join(scan['registry_modules']) if scan['registry_modules'] else "")
        ts_modules_list.append('|'.join(scan['template_spec_modules']) if scan['template_spec_modules'] else "")
        external_files_list.append('|'.join(scan['external_files']) if scan['external_files'] else "")

        # Validate with pycep
        try:
            with open(template_path, 'r', encoding='utf-8-sig') as f:
                content = f.read()
            result = pycep_parser.parse(text=content)
            # result = pycep_parser.parse(file_path=Path(template_path))

            resources = result.get('resources', {})
            if not resources:
                is_valid_list.append(False)
                invalid_reason_list.append("No resource found in parsed output")
                continue

            is_valid_list.append(True)
            invalid_reason_list.append("")

        except Exception as e:
            is_valid_list.append(False)
            invalid_reason_list.append(f"pycep parse error: {str(e)}")

    bicep_df['is_valid'] = is_valid_list
    bicep_df['invalid_reason'] = invalid_reason_list
    bicep_df['local_modules'] = local_modules_list
    bicep_df['registry_modules'] = registry_modules_list
    bicep_df['template_spec_modules'] = ts_modules_list
    bicep_df['external_files'] = external_files_list

    bicep_df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')

    total = len(bicep_df)
    valid = sum(is_valid_list)
    invalid = total - valid
    has_local_mod = sum(1 for v in local_modules_list if v)
    has_registry_mod = sum(1 for v in registry_modules_list if v)
    has_ts_mod = sum(1 for v in ts_modules_list if v)
    has_ext_files = sum(1 for v in external_files_list if v)

    print(f"[Bicep] {valid}/{total} valid ({valid/total*100:.1f}%), {invalid} invalid")
    print(f"  Templates with local modules:          {has_local_mod}")
    print(f"  Templates with registry modules (br:): {has_registry_mod}")
    print(f"  Templates with template specs (ts:):   {has_ts_mod}")
    print(f"  Templates with external file refs:     {has_ext_files}")
    if invalid > 0:
        reasons = bicep_df[bicep_df['is_valid'] == False]['invalid_reason'].value_counts()
        for reason, count in reasons.head(10).items():
            print(f"  - {reason}: {count}")

    return bicep_df

In [3]:
# --- Usage ---
# Update the paths below to match your dataset CSV files.
# Each function reads the input CSV, filters to the relevant language,
# validates, adds label columns, and writes the output CSV.

# CloudFormation
# cfn_df = validate_cloudformation("data/cloudformation_collected_templates_new/dataset_metadata.csv", "data/cloudformation_collected_templates_new/cfn_dataset_validation_result.csv")

# ARM
# arm_df = validate_arm("data/arm_collected_templates_new/dataset_metadata.csv", "data/arm_collected_templates_new/arm_dataset_validation_result.csv")

# Terraform
# process_tf_csv("data/terraform_collected_templates_new/dataset_metadata.csv")
# tf_df = validate_terraform("data/terraform_collected_templates_new/dataset_metadata.csv", "data/terraform_collected_templates_new/terraform_dataset_validation_result.csv")

# Bicep - module
# bicep_df = validate_bicep("data/bicep_collected_templates_new_module/dataset_metadata.csv", "data/bicep_collected_templates_new_module/bicep_dataset_validation_result.csv")

# Bicep - no module
bicep_df = validate_bicep("data/bicep_collected_templates_new/dataset_metadata.csv", "data/bicep_collected_templates_new/bicep_dataset_validation_result.csv")

[Bicep] 24839/26613 valid (93.3%), 1774 invalid
  Templates with local modules:          0
  Templates with registry modules (br:): 0
  Templates with template specs (ts:):   0
  Templates with external file refs:     835
  - No resource found in parsed output: 165
  - pycep parse error: Unexpected token Token('AT', '@') at line 21, column 3.
Expected one of: 
	* RESOURCE
	* QUOTED_STRING
	* QUOTED_INTERPOLATION
	* RBRACE
	* STRING
	* _CPP_COMMENT_NL
Previous tokens: [Token('_CPP_COMMENT_NL', '\n\n')]
: 16
  - pycep parse error: Unexpected token Token('LBRACE', '{') at line 36, column 11.
Expected one of: 
	* RESOURCE
	* QUOTED_STRING
	* QUOTED_INTERPOLATION
	* RBRACE
	* STRING
	* _CPP_COMMENT_NL
Previous tokens: [Token('_CPP_COMMENT_NL', '\n')]
: 16
  - pycep parse error: Unexpected token Token('RPAR', ')') at line 73, column 264.
Expected one of: 
	* __ANON_8
	* LESSTHAN
	* __ANON_6
	* __ANON_0
	* QMARK
	* __ANON_7
	* PERCENT
	* COMMA
	* STAR
	* PLUS
	* __ANON_11
	* SLASH
	* __ANON_1

## Fix Bicep Dataset by Generating Depending Files

In [22]:
_STUB_CONTENT = {
    '.json':  '{}',
    '.jsonc': '{}',
    '.yaml':  '{}',
    '.yml':   '{}',
    '.xml':   '<root/>',
}


def generate_depending_files(
    csv_path: str,
    templates_dir: str = 'data/bicep_collected_templates_new',
    encoding: str = 'utf-8-sig',
    dry_run: bool = False,
) -> pd.DataFrame:
    """
    Create stub files for every external dependency listed in the CSV.

    Bicep functions like loadFileAsBase64, loadTextContent, loadJsonContent,
    and loadYamlContent reference external files that must exist on disk for
    the template to compile. This function reads the ``external_files`` column
    (pipe-separated relative paths), resolves each path relative to the
    template's location, and creates a minimal stub so that downstream
    parsing (az bicep build / pycep) does not fail on missing files.

    Parameters
    ----------
    csv_path : str
        Path to the validated Bicep dataset CSV.  Must contain columns
        ``template_path`` and ``external_files``.
    templates_dir : str
        Directory where the Bicep templates live. Used as a safety boundary:
        stub files that would resolve above this directory are flattened into
        a ``_stubs/`` subdirectory instead.
    encoding : str
        Encoding used to read the CSV (default ``utf-8-sig`` to handle BOM).
    dry_run : bool
        If True, report what *would* be created without writing to disk.

    Returns
    -------
    pd.DataFrame
        A summary DataFrame with columns
        ``[template_name, external_file, resolved_path, status]``.
    """
    df = pd.read_csv(csv_path, encoding=encoding)
    templates_root = Path(templates_dir).resolve()
    stubs_dir = templates_root / '_stubs'

    has_ext = df['external_files'].notna() & (df['external_files'].astype(str).str.strip() != '')
    ext_df = df.loc[has_ext, ['template_name', 'template_path', 'external_files']].copy()

    records: list[dict] = []
    created = 0
    skipped = 0

    for _, row in ext_df.iterrows():
        template_path = Path(row['template_path']).resolve()
        template_dir = template_path.parent
        raw_files = str(row['external_files']).split('|')

        for raw in raw_files:
            raw = raw.strip()
            if not raw:
                continue

            resolved = (template_dir / raw).resolve()

            if not str(resolved).startswith(str(templates_root)):
                safe_name = Path(raw).name
                resolved = (stubs_dir / safe_name).resolve()

            status = ''
            if resolved.exists():
                status = 'already_exists'
                skipped += 1
            elif dry_run:
                status = 'would_create'
            else:
                resolved.parent.mkdir(parents=True, exist_ok=True)
                suffix = resolved.suffix.lower()
                content = _STUB_CONTENT.get(suffix, '')
                resolved.write_text(content, encoding='utf-8')
                status = 'created'
                created += 1

            records.append({
                'template_name': row['template_name'],
                'external_file': raw,
                'resolved_path': str(resolved),
                'status': status,
            })

    summary = pd.DataFrame(records)
    print(f"Templates with external files: {len(ext_df)}")
    print(f"Total external file references: {len(records)}")
    print(f"Unique resolved paths: {summary['resolved_path'].nunique()}")
    print(f"Created: {created}  |  Already existed: {skipped}")
    if dry_run:
        would = int((summary['status'] == 'would_create').sum())
        print(f"Would create (dry_run): {would}")

    return summary

In [23]:
# --- Generate stub files for Bicep external dependencies ---
# Dry run first to preview what will be created
# summary = generate_depending_files(
#     "data/bicep_collected_templates_new/bicep_dataset_validation_result.csv",
#     dry_run=True,
# )
# display(summary)

# Actual creation
summary = generate_depending_files(
    "data/bicep_collected_templates_new/bicep_dataset_validation_result.csv",
)
display(summary['status'].value_counts())

Templates with external files: 835
Total external file references: 1362
Unique resolved paths: 496
Created: 0  |  Already existed: 1362


status
already_exists    1362
Name: count, dtype: int64

## Analysis

In [4]:
def _normalize_invalid_reason(reason: str) -> str:
    """
    Normalize an invalid_reason string to a groupable category.
    Strips variable parts (file paths, line/column numbers, schema values)
    so that structurally identical reasons are grouped together.
    """
    if pd.isna(reason) or not str(reason).strip():
        return ""
    msg = str(reason).strip()

    # Strip file-specific paths from "File not found: ..." and "File(s) not found: ..."
    msg = re.sub(r"(File(?:\(s\))? not found): .+", r"\1", msg)
    # Strip schema value from "Invalid or missing $schema: ..."
    msg = re.sub(r"(Invalid or missing \$schema): .+", r"\1", msg)
    # Strip provider list from "No known cloud provider found (detected: ...)"
    msg = re.sub(r"(No known cloud provider found) \(detected: [^)]+\)", r"\1", msg)
    # Normalize line/column/char references (e.g. "line 3 column 10 (char 49)")
    msg = re.sub(
        r"\s*:?\s*line\s+\d+\s+column\s+\d+(\s*\(char\s+\d+\))?",
        "",
        msg,
        flags=re.IGNORECASE,
    )
    msg = re.sub(r"\s*\(char\s+\d+\)\s*", "", msg)

    return msg.strip()


def evaluate_invalid_reasons(
    validated_csv_path: str,
    output_csv_path: str | None = None,
    encoding: str = 'utf-8-sig',
) -> pd.DataFrame:
    """
    Summarize invalidity reasons from a validated dataset CSV (output of any
    validate_* function).

    Reads the CSV, filters to rows where is_valid == False and invalid_reason
    is non-empty, normalizes each reason into a groupable category, and produces
    a summary DataFrame with counts.

    Parameters
    ----------
    validated_csv_path : str
        Path to a validated CSV that contains 'is_valid' and 'invalid_reason' columns.
    output_csv_path : str, optional
        If provided, write the summary DataFrame to this CSV.
    encoding : str, default 'utf-8-sig'
        Encoding for reading the input CSV.

    Returns
    -------
    pd.DataFrame
        Columns: failure_type (str), count (int), percentage (float).
        Sorted by count descending.
    """
    df = pd.read_csv(validated_csv_path, encoding=encoding)

    total = len(df)
    if 'is_valid' not in df.columns or 'invalid_reason' not in df.columns:
        raise ValueError("CSV must contain 'is_valid' and 'invalid_reason' columns.")

    valid_count = int(df['is_valid'].sum())
    invalid_mask = (df['is_valid'] == False) & df['invalid_reason'].notna() & (df['invalid_reason'].astype(str).str.strip() != "")
    failures = df[invalid_mask].copy()

    print(f"Total templates: {total}")
    print(f"Valid:   {valid_count} ({valid_count / total * 100:.1f}%)")
    print(f"Invalid: {len(failures)} ({len(failures) / total * 100:.1f}%)")

    if failures.empty:
        print("No invalid templates found.")
        return pd.DataFrame({"failure_type": [], "count": [], "percentage": []})

    failures["failure_type"] = failures["invalid_reason"].map(_normalize_invalid_reason)

    summary = (
        failures
        .groupby("failure_type", dropna=False)
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .reset_index(drop=True)
    )
    summary["percentage"] = (summary["count"] / len(failures) * 100).round(2)

    print(f"\nInvalid reason summary ({len(summary)} unique types):")
    for _, row in summary.iterrows():
        print(f"  [{row['count']:>5}] ({row['percentage']:>5.1f}%) {row['failure_type']}")

    if output_csv_path:
        summary.to_csv(output_csv_path, index=False, encoding='utf-8-sig')
        print(f"\nSummary saved to: {output_csv_path}")

    return summary

In [44]:
def relabel_bicep_pycep_errors(
    validated_csv_path: str,
    output_csv_path: str,
    encoding: str = 'utf-8-sig',
) -> pd.DataFrame:
    """
    Re-label Bicep validation results to treat pycep parse errors as valid.

    Templates marked invalid due to pycep parser limitations (encoding issues,
    unsupported syntax, etc.) are re-labelled as valid, since the templates
    themselves are structurally correct. Only templates with
    'No resource found in parsed output' remain invalid.
    """
    df = pd.read_csv(validated_csv_path, encoding=encoding)

    invalid_before = int((df['is_valid'] == False).sum())

    mask = (df['is_valid'] == False) & (df['invalid_reason'] != 'No resource found in parsed output')
    df.loc[mask, 'is_valid'] = True
    df.loc[mask, 'invalid_reason'] = ''

    invalid_after = int((df['is_valid'] == False).sum())
    relabelled = invalid_before - invalid_after

    df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')

    print(f"Invalid before: {invalid_before}")
    print(f"Relabelled to valid: {relabelled}")
    print(f"Invalid after:  {invalid_after} (only 'No resource found')")
    print(f"Saved to: {output_csv_path}")

    return df

In [5]:
# --- Evaluate invalid reasons from validated CSVs ---
# Pass the output CSV from any validate_* function above.

# CloudFormation
# cfn_summary = evaluate_invalid_reasons("data/cloudformation_collected_templates_new/cfn_dataset_validation_result.csv", "data/cloudformation_collected_templates_new/cfn_dataset_validation_summary.csv")

# ARM
# arm_summary = evaluate_invalid_reasons("data/arm_collected_templates_new/arm_dataset_validation_result.csv", "data/arm_collected_templates_new/arm_dataset_validation_summary.csv")

# Terraform
# tf_summary = evaluate_invalid_reasons("data/terraform_collected_templates_new/terraform_dataset_validation_result.csv", "data/terraform_collected_templates_new/terraform_dataset_validation_summary.csv")

# Bicep - module
# bicep_summary = evaluate_invalid_reasons("data/bicep_collected_templates_new_module/bicep_dataset_validation_result.csv", "data/bicep_collected_templates_new_module/bicep_dataset_validation_summary.csv")

# Bicep
bicep_summary = evaluate_invalid_reasons("data/bicep_collected_templates_new/bicep_dataset_validation_result.csv", "data/bicep_collected_templates_new/bicep_dataset_validation_summary.csv")
# relabelled_df = relabel_bicep_pycep_errors(
#     "data/bicep_collected_templates_new/bicep_dataset_validation_result.csv",
#     "data/bicep_collected_templates_new/bicep_dataset_validation_result_relabelled.csv"
# )

Total templates: 26613
Valid:   24839 (93.3%)
Invalid: 1774 (6.7%)

Invalid reason summary (819 unique types):
  [  165] (  9.3%) No resource found in parsed output
  [   16] (  0.9%) pycep parse error: Unexpected token Token('LBRACE', '{') at line 36, column 11.
Expected one of: 
	* RESOURCE
	* QUOTED_STRING
	* QUOTED_INTERPOLATION
	* RBRACE
	* STRING
	* _CPP_COMMENT_NL
Previous tokens: [Token('_CPP_COMMENT_NL', '\n')]
  [   16] (  0.9%) pycep parse error: Unexpected token Token('AT', '@') at line 21, column 3.
Expected one of: 
	* RESOURCE
	* QUOTED_STRING
	* QUOTED_INTERPOLATION
	* RBRACE
	* STRING
	* _CPP_COMMENT_NL
Previous tokens: [Token('_CPP_COMMENT_NL', '\n\n')]
  [   16] (  0.9%) pycep parse error: Unexpected token Token('RPAR', ')') at line 73, column 264.
Expected one of: 
	* __ANON_8
	* LESSTHAN
	* __ANON_6
	* __ANON_0
	* QMARK
	* __ANON_7
	* PERCENT
	* COMMA
	* STAR
	* PLUS
	* __ANON_11
	* SLASH
	* __ANON_12
	* __ANON_4
	* __ANON_10
	* MINUS
	* __ANON_5
	* LSQB
	* __ANON_

## Extract Valid Template Dataset

In [6]:
def extract_valid_template_dataset(
    validated_csv_path: str,
    output_csv_path: str,
    encoding: str = 'utf-8-sig',
) -> pd.DataFrame:
    """
    Read a validated CSV, remove rows where is_valid is False,
    drop the validation label columns, and save the filtered dataset.

    Returns the filtered DataFrame.
    """
    df = pd.read_csv(validated_csv_path, encoding=encoding)

    total = len(df)
    valid_df = df[df['is_valid'] == True].copy()
    removed = total - len(valid_df)

    valid_df.drop(columns=['is_valid', 'invalid_reason'], inplace=True, errors='ignore')

    valid_df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')

    print(f"Total: {total} | Valid: {len(valid_df)} | Removed: {removed}")
    print(f"Saved to: {output_csv_path}")

    return valid_df

In [8]:
# CloudFormation
# valid_df = extract_valid_template_dataset(
#     "data/cloudformation_collected_templates_new/cfn_dataset_validation_result.csv",
#     "data/cloudformation_collected_templates_new/cfn_dataset_filtered.csv"
# )

# Terraform 
# valid_df = extract_valid_template_dataset(
#     "data/terraform_collected_templates_new/terraform_dataset_validation_result.csv",
#     "data/terraform_collected_templates_new/terraform_dataset_filtered.csv"
# )

# ARM
# valid_df = extract_valid_template_dataset(
#     "data/arm_collected_templates_new/arm_dataset_validation_result.csv",
#     "data/arm_collected_templates_new/arm_dataset_filtered.csv"
# )

# Bicep - module
# valid_df = extract_valid_template_dataset(
#     "data/bicep_collected_templates_new_module/bicep_dataset_validation_result.csv",
#     "data/bicep_collected_templates_new_module/bicep_dataset_filtered.csv"
# )

# Bicep 
valid_df = extract_valid_template_dataset(
    "data/bicep_collected_templates_new/bicep_dataset_validation_result.csv",
    "data/bicep_collected_templates_new/bicep_dataset_filtered.csv"
)

Total: 26613 | Valid: 24839 | Removed: 1774
Saved to: data/bicep_collected_templates_new/bicep_dataset_filtered.csv
